### The following code was for checking PMs and comparing COGS runs downstream.

In [1]:
source("~/Rfunctions/helen_functions.R")
library(data.table)
library(dplyr) # needed to collapse gene names in the second function.

setwd("~/HRJ_monocytes/hILCs/rCOGS_in/Version3_revision2/peakmatrices")


Attaching package: ‘dplyr’


The following objects are masked from ‘package:data.table’:

    between, first, last


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




### First, making one big PM for Chicago and ABCC. Then make one without ABCC, for comparison.

In [ ]:
ILC <- fread("./ILC3_chicago_fres_bin_5kb_abc_023_fres_extended_peakm_13012025_modified.txt")
cols_to_remove <- c("chicago_score_fres", "chicago_score_5kb", "ABC.Score", "N_fres", "N_5kb", "N_abc", "baitID_5kb", "oeID_5kb")
print(ILC)

ILC_fres <- ILC[!is.na(chicago_score_fres), ]
ILC_fres[, Score := chicago_score_fres]
ILC_fres[, c(cols_to_remove) := NULL]

ILC_5kb <- ILC[!is.na(chicago_score_5kb), ]
ILC_5kb[, Score := chicago_score_5kb]
ILC_5kb[, c(cols_to_remove) := NULL]

ILC_ABC <- ILC[!is.na(ABC.Score), ]
ILC_ABC[, Score := ABC.Score]
ILC_ABC[, c(cols_to_remove) := NULL]

ILC_chic_ABC <- rbind(ILC_fres, ILC_5kb, ILC_ABC)
print(ILC_chic_ABC)
fwrite_headers(ILC_chic_ABC, file = "./ILC3_chicago_fres_5kb_abc_023_fres_extended_peakm_13012025_modified_oneCol.txt")

ILC_chic <- rbind(ILC_fres, ILC_5kb)
print(ILC_chic)
fwrite_headers(ILC_chic, file = "./ILC3_chicago_fres_5kb_extended_peakm_13012025_modified_oneCol_noABC.txt")

#print(ILC_fres)
#print(ILC_5kb)
#print(ILC_ABC)

Then run the COGS scripts in ~/HRJ_monocytes/hILCs/scripts/helen_scripts_for_rCOGS_in/ABC_thresholds_Jan2025 to get COGS inputs.

#### Below code was for checking the results of ABCC during development: make scatterplot of score threshold 023 versus 022 for imputed classic COGS

In [ ]:
library(data.table)
library(dplyr)
library(ggplot2)
library(ggrepel)

mydir <- "~/HRJ_monocytes/hILCs/COGS_results/"
#mydir <- "~/Documents/analysis/hILCs/"
results <- paste0(mydir, "COGS_out")
outdir = paste0(mydir, "comparisons")
#dir.create(outdir)
list.files(results)

##### To run with pre-annotated results
make_comparison_plot_annotated <- function(res1, name1, res2, name2, outdir, outname, imageType) {
  results1 <- fread(paste(results, res1, sep = "/"))
  results2 <- fread(paste(results, res2, sep = "/"))
  results1[, chr37 := as.character(chr37)]
  results2[, chr37 := as.character(chr37)]
  results1[, chr38 := as.character(chr38)]
  results2[, chr38 := as.character(chr38)]
  
  setkey(results1, ensg, chr37, chr38, gene, minTss37, maxTss37, minTss38, maxTss38)
  
  both <- merge.data.table(results1, results2, on = c("ensg", "chr37", "chr38", "gene", "minTss37", "maxTss37", "minTss38", "maxTss38"), all = TRUE)
  
  both[cogs.x >= 0.5 & cogs.y < 0.5, ':=' (label = gene, 
                                         group = name1)]
  both[cogs.x < 0.5 & cogs.y >= 0.5, ':=' (label = gene, 
                                         group = name2)]
  both[cogs.x >= 0.5 & cogs.y >= 0.5, ':=' (label = gene, 
                                          group = "Both")]

  # Fix NAs 
  both[cogs.x >= 0.5 & is.na(cogs.y), ':=' (group = name1,
                                           label = gene, 
                                           cogs.y = 0)]
  
  both[is.na(cogs.x) & cogs.y >= 0.5, ':=' (group = name2, 
                                            label = gene, 
                                            cogs.x = 0)]

  
  cogs1 <- paste0("cogs_", name1)
  cogs2 <- paste0("cogs_", name2)
  
  setnames(both, "cogs.x", cogs1)
  setnames(both, "cogs.y", cogs2)
  
  # Make a scatter plot.
  if(imageType == "pdf") {
    pdf(file = paste0(outdir, outname, ".pdf"), width = 12, height = 12)
    p <- ggplot(both, aes(x = get(cogs1), y = get(cogs2), label = label, colour = group))
    
    print(p + geom_point() +
            geom_text_repel(aes(label = label),
                            #box.padding   = 0.2, 
                            #point.padding = 0.2,
                            segment.color = 'grey50',
                            size = 4, 
                            max.overlaps = 20) + 
            theme(text = element_text(size = 20), panel.background = element_rect(fill = "white", colour = "grey50"), 
                  legend.position = "bottom") +
            xlab(name1) +
            ylab(name2)) 
    dev.off()
    } else { 
      if(imageType == "jpeg") {
        jpeg(file = paste0(outdir, outname, ".jpg"))
        p <- ggplot(both, aes(x = get(cogs1), y = get(cogs2), label = label, colour = group))
        print(p + geom_point() +
                geom_text_repel(aes(label = label),
                                #box.padding   = 0.2, 
                                #point.padding = 0.2,
                                segment.color = 'grey50',
                                size = 3, 
                                max.overlaps = 20) + 
                xlab(name1) +
                ylab(name2) +
                theme_classic())
        dev.off() 
        } else { if(imageType == "png") {
          png(file = paste0(outdir, outname, ".png"))
          p <- ggplot(both, aes(x = get(cogs1), y = get(cogs2), label = label, colour = group))
          print(p + geom_point() +
                  geom_text_repel(aes(label = label),
                                  #box.padding   = 0.2, 
                                  #point.padding = 0.2,
                                  segment.color = 'grey50',
                                  size = 3, 
                                  max.overlaps = 20) + 
                  xlab(name1) +
                  ylab(name2) +
                  theme_classic())
          dev.off() 
        } else { print("Please specify pdf, jpeg or png")
        }
        }
    }
  return(both)
}

In [ ]:
##### CD4 versus ILC3

setwd(mydir)
cells_comparison <- make_comparison_plot_annotated(res1 = "Version3_revision2/revision_deLange_ILCs_hg38_SuSIE_combinedInteractions_extended_ABC023/Annotated_COGS_scores_data.table.txt", 
                                                   name1 = "ILC_SuSIE_ABC_023_Jan2025", 
                                                   res2 = "Version3_revision2/revision_deLange_CD4s_hg38_SuSIE_fix_combinedInteractions_extended_ABC023/Annotated_COGS_scores_data.table.txt", 
                                                   name2 = "CD4_SuSIE_ABC_023_Jan2025", 
                                                   outdir = paste0(mydir, "comparisons/"), 
                                                   outname = "ILC_CD4_SuSIE_comparisonJan25_scatterplot", 
                                                   imageType = "pdf")

In [ ]:
##### initial submission ILC3 versus revision
setwd(mydir)
ABC_comparison <- make_comparison_plot_annotated(res1 = "Version3_revision2/revision_deLange_ILCs_hg38_SuSIE_combinedInteractions_extended_ABC023/Annotated_COGS_scores_data.table.txt", 
                                                   name1 = "ILC_SuSIE_ABC_023_Jan2025", 
                                                   res2 = "natGen_submission1_ILCs/CD_deLange_ILCs_hg38_SuSIE_combinedInteractions_ALL/Annotated_COGS_scores_data.table.txt", 
                                                   name2 = "ILC_ClassicImpute_ABC_natGen", 
                                                   outdir = paste0(mydir, "comparisons/"), 
                                                   outname = "ILC_SuSIE_ABCcomparison_oldVnew_Jan25_scatterplot", 
                                                   imageType = "pdf")

### 

In [ ]:
#### Checking the result where we use one column

#####
setwd(mydir)
ABC_comparison <- make_comparison_plot_annotated(res1 = "Version3_revision2/revision_deLange_ILCs_hg38_SuSIE_combinedInteractions_extended_ABC023_oneCol/Annotated_COGS_scores_data.table.txt", 
                                                   name1 = "ILC_SuSIE_ABC_023_Jan2025_oneCol", 
                                                   res2 = "Version3_revision2/revision_deLange_ILCs_hg38_SuSIE_combinedInteractions_extended_oneCol_noABC/Annotated_COGS_scores_data.table.txt", 
                                                   name2 = "ILC_SuSIE_Jan2025_oneCol_noABC", 
                                                   outdir = paste0(mydir, "comparisons/"), 
                                                   outname = "ILC_SuSIE_ABCcomparisonJan25_oneCol_scatterplot", 
                                                   imageType = "pdf")

### Note we now do see a difference between ABC and no ABC

In [ ]:
### check for any differences.
print(ABC_comparison)
print(ABC_comparison[cogs_ILC_SuSIE_ABC_023_Jan2025_oneCol != cogs_ILC_SuSIE_Jan2025_oneCol_noABC]) 

In [ ]:
### Now check that we get the same result if we use one column of fragres, 5kb and ABC, compared to using different columns.
### IF it is different, we need to explore.

setwd(mydir)
Column_comparison <- make_comparison_plot_annotated(res1 = "Version3_revision2/revision_deLange_ILCs_hg38_SuSIE_combinedInteractions_extended_ABC023_oneCol/Annotated_COGS_scores_data.table.txt", 
                                                   name1 = "ILC_SuSIE_ABC_023_Jan2025_oneCol", 
                                                   res2 = "Version3_revision2/revision_deLange_ILCs_hg38_SuSIE_combinedInteractions_extended_ABC023_CHiCandABCOnly/Annotated_COGS_scores_data.table.txt", 
                                                   name2 = "ILC_SuSIE_Jan2025_difCols", 
                                                   outdir = paste0(mydir, "comparisons/"), 
                                                   outname = "ILC_SuSIE_ABCcomparisonJan25_oneColvsDifcols_scatterplot", 
                                                   imageType = "pdf")

### check for any differences.
print(Column_comparison)
print(Column_comparison[cogs_ILC_SuSIE_ABC_023_Jan2025_oneCol != cogs_ILC_SuSIE_Jan2025_difCols]) 

# they are exactly the same - great!
# We will use the multi-column method for the paper.

#### Checking the results of modified ABC: make scatterplot of 023 versus 022 for SuSIE results

In [ ]:
setwd(mydir)
SuSIE_comparison <- make_comparison_plot_annotated(res1 = "Version3_revision2/revision_deLange_ILCs_hg38_SuSIE_combinedInteractions_extended_ABC023/Annotated_COGS_scores_data.table.txt", 
                                                   name1 = "ILC_SuSIE_ABC_023_Jan2025", 
                                                   res2 = "natGen_submission1_ILCs/CD_deLange_ILCs_hg38_SuSIE_combinedInteractions_ALL/Annotated_COGS_scores_data.table.txt", 
                                                   name2 = "ILC_SuSIE_ABC_022_natGen", 
                                                   outdir = paste0(mydir, "comparisons/"), 
                                                   outname = "ILC_SuSIE_ABCcomparisonJan25_scatterplot", 
                                                   imageType = "pdf")

#### We gain the following genes in the new analysis (with updated ABC)

CD244
ICAM4
IL10
TNFSF18
MMP9
RBM17
IL15RA
CIITA
SLC12A5
DDC

In [ ]:
### compare with and without ABC

SuSIE_comparison <- make_comparison_plot_annotated(res1 = "Version3_revision2/revision_deLange_ILCs_hg38_SuSIE_combinedInteractions_extended_ABC023/Annotated_COGS_scores_data.table.txt", 
                                                   name1 = "ILC_SuSIE_ABC_023_Jan2025", 
                                                   res2 = "Version3_revision2/revision_deLange_ILCs_hg38_SuSIE_combinedInteractions_extended_ABC023_NoABC/Annotated_COGS_scores_data.table.txt", 
                                                   name2 = "ILC_SuSIE_Jan2025_NoABC", 
                                                   outdir = paste0(mydir, "comparisons/"), 
                                                   outname = "ILC_SuSIE_plusMinusABC_Jan25_scatterplot", 
                                                   imageType = "pdf")

### All the new genes since the first analysis (NatGen submission) are gained because of ABC.

In [ ]:
pm <- fread("~/spivakov/miniPCHiC/hILCs/ILC3/PCHiC/data/ILC3_chicago_fres_bin_5kb_abc_023_fres_extended_peakm_13012025.txt")

print(head(pm))